# MCP 模型上下文协议

## MCP 是什么

MCP 是 Model Context Protocol 的缩写，是连接大模型与外部资源/工具的标准化接口服务。 说的通俗一些mcp就是标准化的function call，只不过这个function call是用于大模型与外部资源/工具之间的交互。 我们知道大模型本身只是对数据进行计算和处理，本身不具备获取外部资源和工具的能力，而mcp为大模型提供了具备调用外部资源和工具的能力，并且mcp服务的诞生可以让大模型自行调用这些资源和工具。 比如大模型没有能力调用我们本地数据库数据，然后使用特定结构的SQL查询获取数据再生成图表，那么我们现在就可以使用mcp来实现这个功能。一个 mcp 负责通过SQL查询获取数据，然后另一个 mcp 负责生成图表,比如图片格式或者HTML格式。

## 我们将构建什么
许多大语言模型目前没有能力获取天气预报和恶劣天气警报。让我们使用 MCP 来解决这个问题！

我们将构建一个服务器，提供两个工具：get-alerts 和 get-forecast。然后我们将服务器连接到 MCP 主机（在本例中是 Claude for Desktop）：

服务器可以连接到任何客户端。我们在这里选择 Claude for Desktop 是为了简单起见，但我们也有关于构建自己的客户端的指南以及此处的其他客户端列表。

由于服务器是本地运行的，MCP 目前只支持桌面主机。远程主机正在积极开发中。


## Core MCP Concepts - MCP 核心概念

MCP 服务器可以提供三种主要类型的功能：

- `Resources`（资源）：客户端可以读取的类似文件的数据（如 API 响应或文件内容） 也就是说我们自己开发好的后端API或者第三方的API接口或者是文件等，可以通过 Resource 来进行读取的。区别在于传统API调用是我们或程序主动请求和处理数据的过程，但是Resource方式则是将数据源注册为标准化URI资源，允许大模型通过统一接口直接访问，无需关心底层实现细节。也就是说我们有一个URL，这个URL可以直接接入数据库某个表的数据，然后大模型就可以直接通过这个URL来获取这个表的数据。并且不会像使用 tools 那样需要我们进行确认。这样的做法丰富了大模型的上下文信息。标准规范是 Resource 只对数据进行读取，不能进行写入。
- `Tools`（工具）：LLM 可以调用的函数（需用户批准） 也就是说我们开发的 MCP 服务可以提供很多函数，这些函数可以让大模型自行调用，比如我们开发了一个获取天气预报的函数，那么大模型就可以自己调用这个函数来获取天气预报信息。那么类比一下就好比我们传统开发后端API一样，我们自己开发了一个获取天气预报的API，然后我们自己调用这个API来获取天气预报信息。只不过这个流程是把我们自己替换成大模型而已。
- `Prompts`（提示）：帮助用户完成特定任务的预写模板 其实就是为Tools提供一个提示词，让大模型可以根据提示词模板进行回答。就比如说我通过天气预报函数过去今天的天气的同时我还想让大模型给我出门穿衣的建议，那么我就可以为这个天气函数提供一个提示词，让大模型可以根据提示词模板进行回答。




## 前置知识要求
本快速入门假设您熟悉：
   
Python    
像 Claude 这样的大语言模型    
系统要求     
安装 Python 3.10 或更高版本。     
您必须使用 Python MCP SDK 1.2.0 或更高版本。     

## 设置您的环境
首先，让我们安装 uv 并设置我们的 Python 项目和环境： uv 是一个 Python 依赖管理工具，类似我们开发node项目需要使用 npm 或 npx 依赖管理工具一样。 那么 Python 的依赖管理工具我们常用的还有 pip 和 venv，比如上期视频我就使用 venv 创建了一个python的虚拟环境去启动一个mcp服务。 只不过 uv 比 pip 和 venv 更快，但实际如何其实我也没多大感受，只不过既然官方文档提供使用uv的方式，那我们就按照官方文档的内容的来做。 至于 uv 与 pip 和 venv 的特点大家可以使用 PPL MCP 服务在cursor中提问获取最新的信息进行比对即可，PPL MCP 服务是我第一个视频讲解到的。

```shell
curl -LsSf https://astral.sh/uv/install.sh | sh
```
